# AJOU-VIS 한국어 STT 벤치마크

1~2주차 과제: STT 후보(faster-whisper small/medium, Whisper transformers, Qwen3-ASR)를
**한국어 인식 정확도(CER) / 지연 / GPU 메모리** 기준으로 비교한다.

**사용법**
1. 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택
2. 위에서부터 순서대로 실행
3. 4번 셀에서 자체 녹음 10문장(wav) 업로드, 5번 셀에서 정답 문장 입력
4. 마지막 셀에서 비교표 확인 + 결과 파일 다운로드

> 주의: Colab에는 마이크가 없으므로 실시간 왕복(`local` 모드) 테스트는 로컬에서 별도로 진행.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 포크 레포 클론 + 의존성 설치 (몇 분 걸림)
!git clone --depth 1 https://github.com/AJOU-VIS/speech-to-speech.git
%cd speech-to-speech
!pip install -q ".[faster-whisper]"
!pip install -q jiwer librosa soundfile
# 설치 후 "RESTART RUNTIME" 버튼이 뜨면 누르지 말고 그대로 진행해보고,
# 다음 셀에서 import 에러가 나는 경우에만 런타임 재시작 후 이 셀부터 다시 실행.

In [ ]:
# 3. 자체 녹음 파일 업로드 (한국어 10문장, wav 권장 / mp3·m4a도 가능)
import os
from google.colab import files

os.makedirs("recordings", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join("recordings", name), "wb") as f:
        f.write(data)
print("업로드된 파일:", sorted(os.listdir("recordings")))

In [1]:
# 4. 정답 문장 입력 — 파일명: 실제 발화 문장 (CER 계산에 사용)
REFERENCES = {
    "sent01.wav": "아주대학교 도서관은 몇 시까지 여나요",
    "sent02.wav": "휴학 신청은 어디에서 하나요",
    "sent03.wav": "복학 신청 기간은 언제인가요?",
    "sent04.wav": "수강신청은 언제부터 할 수 있나요?",
    "sent05.wav": "수강신청을 취소하거나 변경하려면 어떻게 하나요?",
    "sent06.wav": "수강신청 최소·최대 학점은 몇 학점인가요?",
    "sent07.wav": "전공을 변경하거나 복수전공을 신청하려면 어떻게 하나요?",
    "sent08.wav": "재수강은 어떤 과목을 대상으로 할 수 있나요?",
    "sent09.wav": "졸업하려면 총 몇 학점을 이수해야 하나요?",
    "sent10.wav": "학생증을 분실했는데 재발급은 어디에서 신청하나요?"
}

In [ ]:
# 5. 벤치마크 실행
# 후보별로 모델을 한 번만 로드하고, 업로드된 모든 파일을 순서대로 인식한다.
# 측정: 모델 로드 시간, 문장별 추론 시간, 첫 출력까지 지연(TTFT), GPU 메모리 증가량
import gc, time, json
from pathlib import Path
from queue import Queue
from threading import Event

import librosa
import numpy as np
import torch

from speech_to_speech.pipeline.messages import VADAudio

CANDIDATES = [
    {"label": "faster-whisper-small", "kind": "faster-whisper",
     "kwargs": {"model_name": "small", "device": "cuda", "compute_type": "float16",
                "gen_kwargs": {"language": "ko"}}},
    {"label": "faster-whisper-medium", "kind": "faster-whisper",
     "kwargs": {"model_name": "medium", "device": "cuda", "compute_type": "float16",
                "gen_kwargs": {"language": "ko"}}},
    {"label": "whisper-large-v3-turbo", "kind": "whisper",
     "kwargs": {"model_name": "openai/whisper-large-v3-turbo", "device": "cuda",
                "torch_dtype": "float16", "language": "ko"}},
    # Colab 기본 transformers가 qwen3_asr 미지원이면 자동으로 건너뜀
    {"label": "qwen3-asr", "kind": "qwen3-asr",
     "kwargs": {"device": "cuda", "language": "ko"}},
]


def build_handler(kind, kwargs):
    if kind == "faster-whisper":
        from speech_to_speech.STT.faster_whisper_handler import FasterWhisperSTTHandler as H
    elif kind == "whisper":
        from speech_to_speech.STT.whisper_stt_handler import WhisperSTTHandler as H
    elif kind == "qwen3-asr":
        from speech_to_speech.STT.qwen3_asr_handler import Qwen3ASRSTTHandler as H
    else:
        raise ValueError(kind)
    return H(Event(), queue_in=Queue(), queue_out=Queue(), setup_kwargs=dict(kwargs))


def gpu_used_mb():
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1024**2


def transcribe(handler, audio):
    """한 문장을 인식하고 (텍스트, 총 소요시간, 첫 출력까지 지연)을 반환."""
    start = time.perf_counter()
    text, ttft = "", None
    for output in handler.process(VADAudio(audio=audio)):
        if ttft is None:
            ttft = time.perf_counter() - start
        text = getattr(output, "text", str(output))
    return text, time.perf_counter() - start, ttft


audio_files = sorted(p for p in Path("recordings").iterdir()
                     if p.suffix.lower() in {".wav", ".mp3", ".m4a", ".flac"})
assert audio_files, "recordings/ 에 오디오 파일이 없습니다. 3번 셀에서 업로드하세요."
audios = {p.name: librosa.load(p, sr=16000, mono=True)[0].astype(np.float32)
          for p in audio_files}

results = []
for cand in CANDIDATES:
    label = cand["label"]
    print(f"\n===== {label} =====")
    gc.collect(); torch.cuda.empty_cache()
    mem_before = gpu_used_mb()
    try:
        t0 = time.perf_counter()
        handler = build_handler(cand["kind"], cand["kwargs"])  # setup 시 warmup 포함
        load_time = time.perf_counter() - t0
    except Exception as e:
        print(f"  로드 실패, 건너뜀: {e}")
        results.append({"label": label, "status": "failed", "error": str(e)})
        continue

    transcribe(handler, audios[audio_files[0].name])  # 실측 제외용 워밍업 1회

    per_file = []
    for name, audio in audios.items():
        text, elapsed, ttft = transcribe(handler, audio)
        per_file.append({"file": name, "text": text, "seconds": elapsed, "ttft": ttft,
                         "audio_seconds": len(audio) / 16000})
        print(f"  {name} ({elapsed:.2f}s): {text}")

    results.append({"label": label, "status": "ok", "load_time_s": load_time,
                    "gpu_mem_mb": gpu_used_mb() - mem_before, "files": per_file})
    del handler
    gc.collect(); torch.cuda.empty_cache()

with open("stt_benchmark_raw.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("\n원본 결과를 stt_benchmark_raw.json 에 저장했습니다.")

In [ ]:
# 6. CER 계산 + 비교표
import pandas as pd
from jiwer import cer


def normalize(s):
    return " ".join(s.replace(".", "").replace(",", "").replace("?", "").split())


rows, transcripts = [], []
for r in results:
    if r["status"] != "ok":
        rows.append({"모델": r["label"], "상태": "실패"})
        continue
    cers = []
    for f in r["files"]:
        ref = REFERENCES.get(f["file"])
        c = cer(normalize(ref), normalize(f["text"])) if ref else None
        if c is not None:
            cers.append(c)
        transcripts.append({"모델": r["label"], "파일": f["file"], "정답": ref,
                            "인식 결과": f["text"], "CER": c})
    rows.append({
        "모델": r["label"],
        "상태": "ok",
        "평균 CER": round(float(np.mean(cers)), 4) if cers else None,
        "평균 추론(s)": round(float(np.mean([f["seconds"] for f in r["files"]])), 3),
        "평균 TTFT(s)": round(float(np.mean([f["ttft"] for f in r["files"]])), 3),
        "RTF": round(float(np.mean([f["seconds"] / f["audio_seconds"] for f in r["files"]])), 3),
        "모델 로드(s)": round(r["load_time_s"], 1),
        "GPU 메모리(MB)": round(r["gpu_mem_mb"]),
    })

summary = pd.DataFrame(rows)
detail = pd.DataFrame(transcripts)
summary.to_csv("stt_benchmark_summary.csv", index=False)
detail.to_csv("stt_benchmark_detail.csv", index=False)

print("=== 요약 (RTF < 1 이면 실시간보다 빠름) ===")
display(summary)
print("=== 문장별 인식 결과 ===")
display(detail)

In [ ]:
# 7. 결과 파일 다운로드 (지연 측정표 v1 · STT 비교표의 원자료)
from google.colab import files
files.download("stt_benchmark_summary.csv")
files.download("stt_benchmark_detail.csv")
files.download("stt_benchmark_raw.json")